In [1]:
import tensorflow as tf
import numpy as np

# Small text corpus
text = """
deep learning is a branch of machine learning.
transformers use attention to learn relationships in text.
models learn patterns from examples and predict the next token.
machine learning can be applied to many engineering problems.
"""

# Character-level vocabulary
vocab = sorted(set(text))
char_to_id = {c: i for i, c in enumerate(vocab)}
id_to_char = {i: c for c, i in char_to_id.items()}

encoded = np.array([char_to_id[c] for c in text])

# Prepare input and target sequences
seq_length = 40

X, y = [], []

for i in range(len(encoded) - seq_length):
    X.append(encoded[i:i + seq_length])
    y.append(encoded[i + 1:i + seq_length + 1])

X = np.array(X)
y = np.array(y)

dataset = tf.data.Dataset.from_tensor_slices((X, y))
dataset = dataset.shuffle(1000).batch(16).prefetch(tf.data.AUTOTUNE)

# Model parameters
vocab_size = len(vocab)
embedding_dim = 64
num_heads = 2

# Input
inputs = tf.keras.Input(shape=(seq_length,))

# Embeddings
x = tf.keras.layers.Embedding(
    vocab_size, embedding_dim
)(inputs)

# Positional information
positions = tf.range(seq_length)
pos_embedding = tf.keras.layers.Embedding(
    seq_length, embedding_dim
)(positions)

x = x + pos_embedding

# Self-attention
attention = tf.keras.layers.MultiHeadAttention(
    num_heads=num_heads,
    key_dim=embedding_dim // num_heads
)(
    x, x, use_causal_mask=True
)

x = tf.keras.layers.LayerNormalization()(x + attention)

# Feed-forward network
ffn = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(embedding_dim)
])

x = tf.keras.layers.LayerNormalization()(x + ffn(x))

# Predict next character
outputs = tf.keras.layers.Dense(vocab_size)(x)

model = tf.keras.Model(inputs, outputs)

# Compile
model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    )
)

# Train
model.fit(dataset, epochs=20)

Epoch 1/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - loss: 2.7087
Epoch 2/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 2.0172
Epoch 3/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 1.8773
Epoch 4/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 1.8084
Epoch 5/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 1.7709
Epoch 6/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - loss: 1.7346
Epoch 7/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 1.6954
Epoch 8/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - loss: 1.6655
Epoch 9/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 1.6218
Epoch 10/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 1.5799
Epoch 11/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 1.5281
Epoch 12/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 1.4757
Epoch 13/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 1.4212
Epoch 14/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 1.3606
Epoch 15/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 1.2942
Epoc

In [2]:
def generate_text(seed_text, num_chars=200):
    generated = seed_text

    for _ in range(num_chars):
        encoded_input = [
            char_to_id[c] for c in generated[-seq_length:]
        ]

        # Pad to sequence length
        encoded_input = [0] * (seq_length - len(encoded_input)) + encoded_input

        x = np.array([encoded_input])

        # Predict next character
        predictions = model.predict(x, verbose=0)
        next_char_logits = predictions[0, -1, :]

        next_char_id = np.argmax(next_char_logits)
        next_char = id_to_char[next_char_id]

        generated += next_char

    return generated

print(generate_text("deep learning", 200))

deep learningdededed machinsf plern ple ing ing ing ing ine a ang ang be be bra brapplinining bed bed be anining be be a be aplininining a be brang brapplied bed bed ang be braninining be be appplining be a aning 
